# Chunk-Level XGBoost with Worker Effects (Deployment-Safe Features)

Extends the worker effects model by restricting XGBoost to features
available at prediction time only.

**Features excluded (not available at prediction time):**
- Distance: `Travel_Distance`, `log_travel_distance`
- Sequential: `same_aisle`, `same_lockey`, `diff_level`, `same_location`
- Time: `time_of_day`, `day_of_week`, `hour`

**Features retained (always known before chunk starts):**
`Weight`, `Cube`, `Quantity`, `UOM_group`, `Aisle_group`, `Level_group`, `top_100_product`
+ `worker_effect` from mixed model

Steps:
1. Fit mixed effects model on training data → `worker_effect` per worker
2. Train XGBoost with deployment-safe features only
3. Compare baseline vs + worker at chunk level

In [ ]:
from pathlib import Path
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

from feature_engineer import get_engineered_df

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

PATH        = Path("/Users/betsyfrdmn/Lucas_Systems_Capstone_Project/data/processed")
WAREHOUSE   = "OE"
WORKCODES   = ["10", "20", "30"]
MAX_TIME    = 300
BLOCK_SIZE  = 50
RANDOM_STATE = 2026

# Features excluded from XGBoost — not available at prediction time
NOT_AVAILABLE_AT_PREDICTION = [
    "Travel_Distance", "log_travel_distance",   # distance
    "same_aisle", "same_lockey", "diff_level", "same_location",  # sequential
    "time_of_day", "day_of_week", "hour", "Hour",  # time
]


## Helper Functions
Identical to `chunk_model.ipynb` so results are directly comparable.

In [ ]:
def resolve_data_path(warehouse):
    return PATH / f"{warehouse.lower()}_detailed.parquet"


def load_engineered_data(warehouse, workcode, max_time=300):
    d, features_all, cat_cols_all = get_engineered_df(
        file_path=resolve_data_path(warehouse),
        warehouse=warehouse,
        max_time=max_time,
        work_code=str(workcode)
    )
    d = d.copy()
    d["Timestamp"] = pd.to_datetime(d["Timestamp"], errors="coerce")
    d = d.dropna(subset=["Timestamp"]).copy()
    d["date"]     = d["Timestamp"].dt.date
    d["WorkCode"] = d["WorkCode"].astype(str).str.replace(".0", "", regex=False)

    # Strip all features not available at prediction time
    # (distance, sequential, and time-based features)
    features = [
        f for f in features_all
        if f not in NOT_AVAILABLE_AT_PREDICTION
    ]
    cat_cols = [
        c for c in cat_cols_all
        if c not in NOT_AVAILABLE_AT_PREDICTION
    ]

    return d, features, cat_cols


def split_by_days(df, test_ratio=0.15):
    all_days   = sorted(df["date"].dropna().unique())
    n_test     = max(1, int(round(len(all_days) * test_ratio)))
    test_days  = all_days[-n_test:]
    train_df   = df[df["date"] < test_days[0]].copy()
    test_df    = df[df["date"].isin(test_days)].copy()
    return train_df, test_df, test_days


def make_X(train_df, test_df, features, cat_cols):
    X_train = pd.get_dummies(train_df[features], columns=cat_cols, drop_first=True)
    X_test  = pd.get_dummies(test_df[features],  columns=cat_cols, drop_first=True)
    X_test  = X_test.reindex(columns=X_train.columns, fill_value=0)
    X_train = X_train.replace([np.inf, -np.inf], np.nan).fillna(0).astype(float)
    X_test  = X_test.replace([np.inf, -np.inf], np.nan).fillna(0).astype(float)
    return X_train, X_test


def make_test_blocks(test_df, block_size=50):
    """
    Splits each (UserID, date) group into fixed-size consecutive chunks.
    Returns a list of dicts with block metadata and row indices.
    """
    d = test_df.sort_values(["UserID", "Timestamp"]).copy()
    blocks = []
    for (uid, day), g in d.groupby(["UserID", "date"], sort=False):
        g = g.reset_index()
        for start in range(0, len(g) - block_size + 1, block_size):
            chunk = g.iloc[start:start + block_size]
            if len(chunk) == block_size:
                blocks.append({
                    "BlockID": f"{uid}_{day}_{start // block_size}",
                    "UserID":  uid,
                    "date":    day,
                    "indices": chunk["index"].tolist()
                })
    return blocks


def eval_blocks(blocks, actual_series, pred_series):
    """Aggregate task-level predictions to block level and evaluate."""
    actual_blocks = [actual_series.loc[b["indices"]].sum() for b in blocks]
    pred_blocks   = [pred_series.loc[b["indices"]].sum() for b in blocks]
    mae  = mean_absolute_error(actual_blocks, pred_blocks)
    r2   = r2_score(actual_blocks, pred_blocks)
    return mae, r2, actual_blocks, pred_blocks


## Step 1: Estimate Worker Effects via Mixed Model

Fit `Time_Delta_sec ~ 1 + (1|UserID)` on the **training set** for each WorkCode separately.
This gives a `worker_effect` b_j per worker — estimated purely from training data to avoid leakage.

The effect is then joined onto both train and test rows, and aggregated (mean) to the chunk level.

In [ ]:
def estimate_worker_effects(train_df):
    """
    Fits a random intercept model on training data.
    Returns a DataFrame with columns [UserID, worker_effect].
    worker_effect is the b_j estimate: positive = slower than average.
    """
    df_re = train_df[["UserID", "Time_Delta_sec"]].dropna().copy()

    # Need at least 2 groups to fit mixed model
    if df_re["UserID"].nunique() < 2:
        print("  [Warning] Not enough workers for mixed model — skipping worker effects")
        return pd.DataFrame({"UserID": df_re["UserID"].unique(), "worker_effect": 0.0})

    model  = smf.mixedlm("Time_Delta_sec ~ 1", data=df_re, groups=df_re["UserID"])
    result = model.fit(reml=True, disp=False)

    worker_effects = pd.DataFrame({
        "UserID":        list(result.random_effects.keys()),
        "worker_effect": [float(v.iloc[0]) for v in result.random_effects.values()]
    })

    icc = result.cov_re.values[0][0] / (result.cov_re.values[0][0] + result.scale)
    print(f"  Grand mean: {result.fe_params['Intercept']:.1f}s | "
          f"Worker SD: {np.sqrt(result.cov_re.values[0][0]):.1f}s | "
          f"ICC: {icc:.3f}")

    return worker_effects

## Step 2: Main Loop — Train, Predict, Evaluate at Chunk Level

In [ ]:
all_block_results = []
all_block_detail  = []

for wc in WORKCODES:
    print(f"\n{'='*55}")
    print(f"WorkCode {wc}")
    print(f"{'='*55}")

    df_wc, features, cat_cols = load_engineered_data(
        warehouse=WAREHOUSE, workcode=wc, max_time=MAX_TIME
    )

    train_df, test_df, test_days = split_by_days(df_wc)
    print(f"Train rows: {len(train_df)} | Test rows: {len(test_df)}")

    y_train = train_df["Time_Delta_sec"].astype(float)
    y_test  = test_df["Time_Delta_sec"].astype(float)

    # --------------------------------------------------
    # Estimate worker effects from training data only
    # --------------------------------------------------
    print("  Fitting mixed model...")
    worker_effects = estimate_worker_effects(train_df)

    # Join worker_effect — unseen workers get 0 (grand mean fallback)
    train_df = train_df.merge(worker_effects, on="UserID", how="left")
    test_df  = test_df.merge(worker_effects,  on="UserID", how="left")
    train_df["worker_effect"] = train_df["worker_effect"].fillna(0.0)
    test_df["worker_effect"]  = test_df["worker_effect"].fillna(0.0)

    # Reset index — critical for block index alignment
    train_df = train_df.reset_index(drop=True)
    test_df  = test_df.reset_index(drop=True)
    y_train  = train_df["Time_Delta_sec"].astype(float)
    y_test   = test_df["Time_Delta_sec"].astype(float)

    # --------------------------------------------------
    # Deployment-safe feature sets only
    # Excludes: distance, sequential, time features
    # --------------------------------------------------
    cats_clean  = [c for c in cat_cols  if c != "efficient_user"]
    feats_clean = [f for f in features  if f != "efficient_user"]

    feats_base = feats_clean
    feats_enh  = feats_clean + ["worker_effect"]

    # --------------------------------------------------
    # Train models
    # --------------------------------------------------
    xgb_params = dict(
        n_estimators=800, learning_rate=0.05, max_depth=6,
        subsample=0.8, colsample_bytree=0.8,
        random_state=RANDOM_STATE, n_jobs=-1
    )

    scenarios = {
        "XGBoost (baseline)": (feats_base, cats_clean),
        "XGBoost (+ worker)": (feats_enh,  cats_clean),
    }

    test_preds = {}
    for label, (feats, cats) in scenarios.items():
        X_train, X_test = make_X(train_df, test_df, feats, cats)
        model = XGBRegressor(**xgb_params)
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        test_preds[label] = pd.Series(preds, index=test_df.index)

    # --------------------------------------------------
    # Build blocks and evaluate at chunk level
    # --------------------------------------------------
    blocks = make_test_blocks(test_df, block_size=BLOCK_SIZE)
    print(f"  Blocks of size {BLOCK_SIZE}: {len(blocks)}")

    for label, pred_series in test_preds.items():
        mae, r2, actual_b, pred_b = eval_blocks(blocks, y_test, pred_series)
        all_block_results.append({
            "WorkCode":     wc,
            "Model":        label,
            "n_blocks":     len(blocks),
            "mae":          round(mae, 3),
            "mae_per_task": round(mae / BLOCK_SIZE, 3),
            "r2":           round(r2, 4),
        })

    for b in blocks:
        row = {
            "BlockID":     b["BlockID"],
            "UserID":      b["UserID"],
            "date":        b["date"],
            "WorkCode":    wc,
            "actual_time": y_test.loc[b["indices"]].sum(),
        }
        for label, pred_series in test_preds.items():
            row[label] = pred_series.loc[b["indices"]].sum()
        all_block_detail.append(row)

block_results_df = pd.DataFrame(all_block_results)
block_detail_df  = pd.DataFrame(all_block_detail)
print("\nDone.")


## Step 3: Results Table

In [ ]:
print(f"Warehouse: {WAREHOUSE} | Block size: {BLOCK_SIZE} tasks\n")

display(
    block_results_df
    .sort_values(["WorkCode", "mae_per_task"])
    .reset_index(drop=True)
)

## Step 4: Impact of Worker Effect — Head-to-Head Comparison

In [ ]:
# Pivot to compare baseline vs +worker side by side
pairs = [
    ("XGBoost (baseline)", "XGBoost (+ worker)"),
]

rows = []
for base_label, enh_label in pairs:
    for wc in WORKCODES:
        base = block_results_df[
            (block_results_df["WorkCode"] == wc) &
            (block_results_df["Model"] == base_label)
        ]
        enh = block_results_df[
            (block_results_df["WorkCode"] == wc) &
            (block_results_df["Model"] == enh_label)
        ]
        if base.empty or enh.empty:
            continue

        mae_base = base["mae_per_task"].values[0]
        mae_enh  = enh["mae_per_task"].values[0]
        improvement_s   = mae_base - mae_enh
        improvement_pct = (improvement_s / mae_base) * 100 if mae_base > 0 else 0

        rows.append({
            "Scenario":             base_label.replace("XGBoost ", "").replace(" baseline", ""),
            "WorkCode":             wc,
            "MAE/task baseline (s)": mae_base,
            "MAE/task + worker (s)": mae_enh,
            "Improvement (s)":      round(improvement_s, 3),
            "Improvement (%)": round(improvement_pct, 2),
        })

display(pd.DataFrame(rows))

## Step 5: Actual vs Predicted Plot (WC 20 & 30, No-Distance + Worker Effect)

In [ ]:
plot_model = "XGBoost (+ worker)"
plot_df = block_detail_df[block_detail_df["WorkCode"].isin(["20", "30"])].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, wc in zip(axes, ["20", "30"]):
    sub = plot_df[plot_df["WorkCode"] == wc]
    ax.scatter(sub["actual_time"], sub[plot_model], alpha=0.6, s=30)
    lims = [min(sub["actual_time"].min(), sub[plot_model].min()),
            max(sub["actual_time"].max(), sub[plot_model].max())]
    ax.plot(lims, lims, "r--", linewidth=1.5, label="Perfect prediction")
    ax.set_xlabel("Actual total time (s) per block")
    ax.set_ylabel("Predicted total time (s) per block")
    ax.set_title(f"WC {wc} — {plot_model}")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle(f"Warehouse {WAREHOUSE} | Block size = {BLOCK_SIZE} tasks", fontsize=13)
plt.tight_layout()
plt.show()